# KKBOX Churn Prediction — Model Training (Memory Optimized)

This is a **memory-optimized version** that skips user_logs features to avoid kernel crashes.

**Changes from original:**
- Skip user_logs_v2.csv (1.4GB file)
- Use only transaction and member features
- Faster training, lower memory usage

## Step 0 — Configuration & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (log_loss, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

TARGET       = 'is_churn'
ID_COL       = 'msno'
TEST_SIZE    = 0.2
RANDOM_STATE = 42

pd.set_option('display.max_columns', None)
print('All libraries loaded successfully.')

## Step 1 — Load Data (Memory Optimized)

In [ ]:
print("Loading data...")
train        = pd.read_csv("../data/train_v2.csv")
members      = pd.read_csv("../data/members_v3.csv")
transactions = pd.read_csv("../data/transactions_v2.csv")

print(f"Train: {train.shape}, Members: {members.shape}, Transactions: {transactions.shape}")

## Step 2 — Preprocessing (Without user_logs)

In [ ]:
# Clean members
members["bd"]     = members["bd"].clip(10, 80)
members["gender"] = members["gender"].fillna("unknown")

# Clean transactions
transactions = transactions.drop_duplicates()
transactions["actual_amount_paid"] = transactions["actual_amount_paid"].clip(0, 5000)
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str), format="%Y%m%d", errors="coerce")

# Discount & expiry-transaction interval
transactions["discount"]            = transactions["plan_list_price"] - transactions["actual_amount_paid"]
transactions["discount_rate"]        = transactions["discount"] / (transactions["plan_list_price"] + 1)
transactions["expiry_txn_interval"]  = (transactions["membership_expire_date"] - transactions["transaction_date"]).dt.days

# Basic transaction aggregations
trans_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["mean", "sum"],
    "payment_plan_days":  ["mean"],
    "is_cancel":          ["sum"],
    "is_auto_renew":      ["mean"]
})
trans_agg.columns = ["_".join(c) for c in trans_agg.columns]
trans_agg = trans_agg.reset_index()
trans_agg["cancel_rate"]     = trans_agg["is_cancel_sum"]          / (trans_agg["payment_plan_days_mean"] + 1)
trans_agg["payment_per_day"] = trans_agg["actual_amount_paid_sum"] / (trans_agg["payment_plan_days_mean"] + 1)

# Advanced transaction aggregations
adv_agg = transactions.groupby("msno").agg({
    "discount":            ["mean", "sum"],
    "discount_rate":       ["mean"],
    "expiry_txn_interval": ["mean", "max"],
    "transaction_date":    ["count"],
})
adv_agg.columns = ["_".join(c) for c in adv_agg.columns]
adv_agg = adv_agg.reset_index().rename(columns={"transaction_date_count": "total_transactions"})

# Last transaction features
last_txn = (
    transactions.sort_values("transaction_date")
    .groupby("msno").tail(1)
    [["msno","is_cancel","is_auto_renew","payment_plan_days",
      "actual_amount_paid","discount","expiry_txn_interval"]]
    .rename(columns={
        "is_cancel":           "last_is_cancel",
        "is_auto_renew":       "last_is_auto_renew",
        "payment_plan_days":   "last_plan_days",
        "actual_amount_paid":  "last_amount_paid",
        "discount":            "last_discount",
        "expiry_txn_interval": "last_expiry_interval",
    })
)

# Temporal features
REFERENCE_DATE = pd.to_datetime("2017-03-01")
last_trans = (
    transactions.sort_values("membership_expire_date")
    .groupby("msno").tail(1)
    [["msno", "membership_expire_date", "transaction_date"]]
)
last_trans["days_left"]             = (last_trans["membership_expire_date"] - REFERENCE_DATE).dt.days
last_trans["days_since_last_txn"]   = (REFERENCE_DATE - last_trans["transaction_date"]).dt.days
temporal_feat = last_trans[["msno", "days_left", "days_since_last_txn"]]

print("\n⚠️  SKIPPING user_logs features to avoid memory issues")
logs_feat = None

# Merge all
df = train.merge(members,      on="msno", how="left")
df = df.merge(trans_agg,        on="msno", how="left")
df = df.merge(adv_agg,          on="msno", how="left", suffixes=("","_adv"))
df = df.merge(last_txn,         on="msno", how="left")
df = df.merge(temporal_feat,    on="msno", how="left")

# Encode & fill
df = pd.get_dummies(df, columns=["city", "gender", "registered_via"], dummy_na=True)
df = df.fillna(0)

print(f"\nFeature matrix: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Features: {df.shape[1]-2} (excl. msno, is_churn)")

## Step 3 — Train/Val Split

In [ ]:
X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Training set   : {X_train.shape[0]:,} rows')
print(f'Validation set : {X_val.shape[0]:,} rows')
print(f'Features       : {X_train.shape[1]}')
print(f'\nChurn rate in train : {y_train.mean():.4f}')
print(f'Churn rate in val   : {y_val.mean():.4f}')

## Step 4 — XGBoost Training

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

xgb = XGBClassifier(
    n_estimators     = 200,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    eval_metric      = 'logloss',
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

xgb_proba   = xgb.predict_proba(X_val)[:, 1]
xgb_logloss = log_loss(y_val, xgb_proba)
xgb_auc     = roc_auc_score(y_val, xgb_proba)

print('\nXGBoost Results:')
print(f'  Log Loss : {xgb_logloss:.4f}')
print(f'  ROC-AUC  : {xgb_auc:.4f}')

## Step 5 — Generate Submission

In [ ]:
test = pd.read_csv("../data/sample_submission_v2.csv")
print(f"Test set shape: {test.shape}")

# Apply same preprocessing
df_test = test[["msno"]].merge(members,   on="msno", how="left")
df_test = df_test.merge(trans_agg,         on="msno", how="left")
df_test = df_test.merge(adv_agg,           on="msno", how="left", suffixes=("","_adv"))
df_test = df_test.merge(last_txn,          on="msno", how="left")
df_test = df_test.merge(temporal_feat,     on="msno", how="left")

df_test = pd.get_dummies(df_test, columns=["city", "gender", "registered_via"], dummy_na=True)
df_test = df_test.fillna(0)

# Align columns
train_cols = X_train.columns.tolist()
for col in train_cols:
    if col not in df_test.columns:
        df_test[col] = 0
X_test = df_test[train_cols]

print(f"Test feature matrix: {X_test.shape}")

# Predict
test_proba = xgb.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "msno":     test["msno"],
    "is_churn": test_proba
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Predicted churn rate: {test_proba.mean():.4f}")
display(submission.head())

submission.to_csv("../data/submission_lite.csv", index=False)
print("\n✅ Saved to ../data/submission_lite.csv")